In [ ]:
import numpy as np

from tqdm import tqdm

from loka.data import load_breakthrough_states, save_breakthrough_states
from loka.models import BreakthroughCNN
from loka.geometrical_methods import herding_order, k_center_order
from loka.embeddings import with_counts, with_near_goal, with_counts_and_near_goal

import os

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
filename = "/home/axelbjarkar/RU/T-404-LOKA-code/data/o.txt"
X,Y = load_breakthrough_states(filename, two_channel=True)

In [ ]:
data_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data"
output_dir = "/home/axelbjarkar/RU/T-404-LOKA-code/data/ordered-data/embeddings"

os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

ratio = 0.10

subset_methods = {
    #'herding':          lambda E: herding_order(E, ratio, device),
    #'k_center_no_S0':   lambda E: k_center_order(E, ratio, device, S0=None),
    'k_center_rand_S0': lambda E: k_center_order(E, ratio, device, S0='rand'),
}
embedding_methods = [
    #with_counts, 
    #with_near_goal, 
    with_counts_and_near_goal
]

orders = []

for subset_name, subset_fn in subset_methods.items():
    for emb in embedding_methods:
        orders.append(
            (
                f"{subset_name}_{emb.__name__}_{round(ratio * 100)}p.txt",
                lambda D, subset_fn=subset_fn, emb=emb: subset_fn(emb(D)),
            )
        )

orders

In [ ]:
for fname, method in orders:
    path = os.path.join(output_dir, fname)

    order = method(X)
    save_breakthrough_states(path, X[order], Y[order])